# v9_b Step 3: Khipu annotation of features to tentative metabolites

- Run Khipu on selected features to get neutral mass values
- Annotate features by ions

In [ ]:
import os
import json

from khipu.extended import peaklist_to_khipu_list, export_empCpd_khipu_list
from asari.tools import match_features as mf

ISOTOPE_RATIO_THRESHOLD = 0.5   # max 13C/12C

outdir = 'results/step3_khipus/'
os.makedirs(outdir, exist_ok=True)

In [ ]:
def deduplicate_features(features):
    seen = set()
    unique_features = {}
    for f in features:
        key = f['id']
        if key not in seen:
            seen.add(key)
            unique_features[key] = f
        else:
            unique_features[key]['enzyme'].extend(f['enzyme'])
            unique_features[key]['substrate'].extend(f['substrate'])
    return unique_features

def convert_line_to_dict(line, substrate, source):
    # unlabeled_feature	unlabeled_feature_mz	unlabeled_feature_rtime	enzyme	unlabeled_treatment_mean_normalized_raw	enzyme_mean_normalized_raw
    a = line.rstrip().split('\t')
    return {
        'id': a[0], 
        'mz': float(a[1]), 
        'rtime': float(a[2]), 
        'representative_intensity': float(a[4]),
        'enzyme': [a[3]], 
        'substrate': [substrate],
        'source': source
    }
    
def get_13c_ratio(kp):
    '''Only consider M+H or M+Na adducts for isotope ratio calculation.
    Find ("ion_relation": "M0,M+H+", "ion_relation": "13C/12C,M+H+",).
    If not, find ("ion_relation": "M0,Na/H", "ion_relation": "13C/12C,Na/H",).
    '''
    features = kp['MS1_pseudo_Spectra']
    M0 = [f for f in features if f.get('ion_relation') == 'M0,M+H+']
    M1 = [f for f in features if f.get('ion_relation') == '13C/12C,M+H+']
    if not M0 or not M1:
        M0 = [f for f in features if f.get('ion_relation') == 'M0,Na/H']
        M1 = [f for f in features if f.get('ion_relation') == '13C/12C,Na/H']
    if not M0 or not M1:
        return 999
    else:
        return M1[0]['representative_intensity'] / M0[0]['representative_intensity']

In [ ]:
# Summarize candidates
'''
unlabeled_feature	unlabeled_feature_mz	unlabeled_feature_rtime	enzyme
F10228	520.2155	750.71	CYP1A1
F10229	520.2155	785.37	CYP1A1
F10232	520.2155	992.06	CYP1A1
F10235	520.2155	1036.84	CYP1A1
'''
dict_04_candidates = {}

v7_step1_results = 'results/step1_filtered_results/'
candidates_files = [f for f in os.listdir(v7_step1_results) if f.startswith('04_v7_')]
count = 0
for f in candidates_files:
    d = []
    for line in open(v7_step1_results + f).readlines()[1:]:
        a = line.rstrip().split('\t')
        d.append({
            'id': a[0], 'mz': float(a[1]), 'rtime': float(a[2]), 'enzyme': a[3]
        })
    c = len(d)
    count += c
    print(c, f)
    dict_04_candidates[f] = d

print("\nTotal features (redundant): ", count)

The above number matches to previous result `03_CYP_steroid_feature_annotations.tsv`. 
Features can be redundant btw substrates, btw cells and supernatant.

## combine features

In [ ]:
# Sanity check, how many merged features
def is_overlap_json_features(F1, F2, mz_tol_ppm=10, rtime_tol=2):
    if abs(F1['mz'] - F2['mz']) / F1['mz'] * 1e6 < mz_tol_ppm and abs(F1['rtime'] - F2['rtime']) < rtime_tol:
        return True
    else:
        return False

DnHz_pellets = [
    '04_v7_dhea_DnHz_pellets_unique_enzyme_specific_unlabeled_features.tsv',
    '04_v7_progesterone_DnHz_pellets_unique_enzyme_specific_unlabeled_features.tsv',
    '04_v7_pregnenolone_DnHz_pellets_unique_enzyme_specific_unlabeled_features.tsv'
]
DnHz_supernatant = [
    '04_v7_dhea_DnHz_supernatant_unique_enzyme_specific_unlabeled_features.tsv',
    '04_v7_progesterone_DnHz_supernatant_unique_enzyme_specific_unlabeled_features.tsv',
    '04_v7_pregnenolone_DnHz_supernatant_unique_enzyme_specific_unlabeled_features.tsv'
]
DnCl_pellets = [
    '04_v7_estrone_DnCl_pellets_unique_enzyme_specific_unlabeled_features.tsv',
    '04_v7_estradiol_DnCl_pellets_unique_enzyme_specific_unlabeled_features.tsv'
]
DnCl_supernatant = [
    '04_v7_estrone_DnCl_supernatant_unique_enzyme_specific_unlabeled_features.tsv',
    '04_v7_estradiol_DnCl_supernatant_unique_enzyme_specific_unlabeled_features.tsv'
]   

DnHz_pellets_candidates, DnHz_supernatant_candidates, DnCl_pellets_candidates, DnCl_supernatant_candidates = [], [], [], []

for f in DnHz_pellets:
    substrate = f.split('_')[2]
    for line in open(v7_step1_results + f).readlines()[1:]:
        DnHz_pellets_candidates.append(convert_line_to_dict(line, substrate, source='DnHz_pellets'))
for f in DnHz_supernatant:
    substrate = f.split('_')[2]
    for line in open(v7_step1_results + f).readlines()[1:]:
        DnHz_supernatant_candidates.append(convert_line_to_dict(line, substrate, source='DnHz_supernatant'))

for f in DnCl_pellets:
    substrate = f.split('_')[2]
    for line in open(v7_step1_results + f).readlines()[1:]:
        DnCl_pellets_candidates.append(convert_line_to_dict(line, substrate, source='DnCl_pellets'))
for f in DnCl_supernatant:
    substrate = f.split('_')[2]
    for line in open(v7_step1_results + f).readlines()[1:]:
        DnCl_supernatant_candidates.append(convert_line_to_dict(line, substrate, source='DnCl_supernatant'))

print(len(DnHz_pellets_candidates), len(DnHz_supernatant_candidates), 
      len(DnCl_pellets_candidates), len(DnCl_supernatant_candidates))

# deduplicate features
DnHz_pellets_candidates = deduplicate_features(DnHz_pellets_candidates)
DnHz_supernatant_candidates = deduplicate_features(DnHz_supernatant_candidates)
DnCl_pellets_candidates = deduplicate_features(DnCl_pellets_candidates)
DnCl_supernatant_candidates = deduplicate_features(DnCl_supernatant_candidates)

print(len(DnHz_pellets_candidates), len(DnHz_supernatant_candidates), 
      len(DnCl_pellets_candidates), len(DnCl_supernatant_candidates))


In [ ]:
DnCl_supernatant_candidates['F11701']

In [ ]:
# pos ionization
isotope_search_patterns_pos = [ (1.003355, '13C/12C', (0, 0.8)),]
adduct_search_patterns_pos = [  # initial patterns are relative to M+H+
                            (21.98194, 'Na/H'),
                            # (41.026549, 'ACN'),     # Acetonitrile
                            # (67.987424, 'NaCOOH'),
                            # (37.955882, 'K/H'),
                            # (32.026215, 'CH3OH'), 
                            # (17.02655, 'NH3'),
                            # (18.0106, 'H2O'), 
                            # (-17.02655, '-NH3'),
                            # (-18.0106, '-H2O'), 
                            ]

In [ ]:

# 
fdicts = [DnHz_pellets_candidates, DnHz_supernatant_candidates, DnCl_pellets_candidates, DnCl_supernatant_candidates]
labels = ['DnHz_pellets', 'DnHz_supernatant', 'DnCl_pellets', 'DnCl_supernatant']

for ii in range(len(fdicts)):
    list_features = list(fdicts[ii].values())
    list_khipus, all_assigned_fids = peaklist_to_khipu_list(
                            list_features, 
                            isotope_search_patterns=isotope_search_patterns_pos, 
                            adduct_search_patterns=adduct_search_patterns_pos,
                            extended_adducts=[], 
                            mz_tolerance_ppm=5,
                            rt_tolerance=2,
                            mode='pos',
                            charges=[1], #, 2, 3], 
                            )

    # convert khipu instances to dicts
    list_khipus = export_empCpd_khipu_list(list_khipus) 
    kd = {}
    for kp in list_khipus:
        # check isotope ratio
        _ratio = get_13c_ratio(kp)
        if _ratio < ISOTOPE_RATIO_THRESHOLD:
            for f in kp['MS1_pseudo_Spectra']:
                f['neutral_formula_mass'] = kp['neutral_formula_mass']
                f['khipu'] = kp['interim_id']
                kd[f['id']] = f
        else:
            print(f"Skipping khipu {kp['interim_id']} due to failed 13C/12C ratio {_ratio}.")
            for f in kp['MS1_pseudo_Spectra']:
                f['ion_relation'] = ''
            
    print(len(list_features), len(list_khipus), len(all_assigned_fids))
    for f in list_features:
        if f['id'] in kd:
            f['neutral_formula_mass'] = kd[f['id']]['neutral_formula_mass']
            f['khipu'] = kd[f['id']]['khipu']
            
    json.dump(list_khipus, open(outdir + labels[ii] + '_ecpds.json', 'w'), indent=4)
    json.dump(list_features, open(outdir + labels[ii] + '_features.json', 'w'), indent=4)


In [ ]:
deu = DEUTERIUM_MASS_SHIFT = 1.006276
C13 = 1.003355

In [ ]:
535.234 - 534.2307

In [ ]:
551.2288 - 550.2258, 572.2081 - 550.2258

## Summary

In each list of features, redundant ions (ion annotations) are marked in JSON. 
